In [1]:
import pickle
import math
import numpy as np
import seaborn as sns

from rl_firm.abm.parameter import ModelParameters, FirmParameters, HouseholdParameters
from rl_firm.abm.firm import Firm, LearningFirm
from rl_firm.abm.household import Household
from rl_firm.abm.model import BaselineModel

NOTE: Creating burned-in snapshots of ABMs to facilitate training/evaluating

There are 2 snapshot version:
- One learning firm for training + firm level evaluation
- 100 learning firms for macro analysis (learning firm can use baseline heuristics for different adoption scenarios)

## Training + firm level evaluation (1 learning firm)

In [3]:
parameters = ModelParameters(
    month_length = 21,
    initial_employment_rate = 0.0,
    n_firms = 100,
    n_learning_firms = 1,
    n_households = 1000,
    burn_in_months = 1000,
    
    firm = FirmParameters(
        gamma_consecutive_months = 24,
        delta_wage_adjustment_rate = 0.019,
        upper_phi_demand = 1,
        lower_phi_demand = 0.25,
        upper_phi_marginal_cost = 1.15,
        lower_phi_marginal_cost = 1.025,
        theta_goods_price = 0.02,
        theta_probs_set_new_price = 0.75,
        lambda_technology = 3,
        chi_buffer = 0.1,

        initial_open_position = True,
        initial_liquidity = 0,
        initial_goods_price = 40,
        initial_wage_rate = 2000,
        #initial_goods_price = 10,
        #initial_wage_rate = 600,
    ),

    household = HouseholdParameters(
        xi_price_percentage = 0.01,
        beta_firm_count = 5,
        pi_explore_odd = 0.1,
        alpha_budget_exponent = 0.9,
        n_suppliers = 7,
        psi_price = 0.25,
        psi_quant = 0.25,

        initial_reservation_wage = 0,
        initial_liquidity = 5000,
    )
)

In [9]:
models = [BaselineModel(parameters=parameters, seed=seed) for seed in range(40)]

In [4]:
%%time
model.burn_in()

CPU times: user 1min 36s, sys: 30.7 ms, total: 1min 36s
Wall time: 1min 36s


In [8]:
%%time

[model.burn_in() for model in models]

CPU times: user 16min 41s, sys: 3.06 s, total: 16min 44s
Wall time: 17min 17s


[None, None, None, None, None, None, None, None, None, None]

In [9]:
[[(f.unique_id, f.profit_ranking, round(f.goods_price, 2), round(f.wage_rate, 2), round(f.last_month_profit, 2), round(f.inventories, 2), f.employee_count, f.customer_count) for f in model.firms[-1:]] for model in models]

[[(100, 43, 42.93, 2186.25, 5055.7, 30.0, 10, 239)],
 [(100, 100, 48.06, 2408.92, -2612.26, 302.56, 11, 46)],
 [(100, 97, 41.82, 2068.56, 0.0, 0.0, 0, 6)],
 [(100, 99, 45.32, 948.29, 0.0, 0.0, 0, 3)],
 [(100, 37, 44.69, 2353.14, 4954.52, 33.0, 11, 38)],
 [(100, 20, 43.43, 2369.64, 6467.26, 54.0, 18, 83)],
 [(100, 66, 40.89, 2156.01, 1137.98, 9.0, 3, 31)],
 [(100, 15, 43.18, 2364.33, 7793.19, 87.91, 23, 82)],
 [(100, 58, 47.79, 2612.55, 1596.54, 31.57, 6, 29)],
 [(100, 83, 45.48, 2242.78, 486.2, 3.0, 1, 22)]]

In [10]:
for model in models:
    # Store the burned in model as pickle to reuse it
    with open(f"../snapshots/abm_{model._seed}.pkl", "wb") as f:
        pickle.dump(model, f)

## Macro analysis model

In [11]:
parameters = ModelParameters(
    month_length = 21,
    initial_employment_rate = 0.0,
    n_firms = 100,
    n_learning_firms = 100,
    n_households = 1000,
    burn_in_months = 1000,
    
    firm = FirmParameters(
        gamma_consecutive_months = 24,
        delta_wage_adjustment_rate = 0.019,
        upper_phi_demand = 1,
        lower_phi_demand = 0.25,
        upper_phi_marginal_cost = 1.15,
        lower_phi_marginal_cost = 1.025,
        theta_goods_price = 0.02,
        theta_probs_set_new_price = 0.75,
        lambda_technology = 3,
        chi_buffer = 0.1,

        initial_open_position = True,
        initial_liquidity = 0,
        initial_goods_price = 40,
        initial_wage_rate = 2000,
    ),

    household = HouseholdParameters(
        xi_price_percentage = 0.01,
        beta_firm_count = 5,
        pi_explore_odd = 0.1,
        alpha_budget_exponent = 0.9,
        n_suppliers = 7,
        psi_price = 0.25,
        psi_quant = 0.25,

        initial_reservation_wage = 0,
        initial_liquidity = 5000,
    )
)

In [12]:
macro_models = [BaselineModel(parameters=parameters, seed=seed) for seed in range(20, 40)]

In [14]:
len(macro_models[0].learning_firms)

100

In [15]:
%%time

[macro_model.burn_in() for macro_model in macro_models]

CPU times: user 32min 44s, sys: 7.16 s, total: 32min 51s
Wall time: 33min 18s


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [17]:
for macro_model in macro_models:
    # Store the burned in model as pickle to reuse it
    with open(f"../snapshots/macro/abm_{macro_model._seed}.pkl", "wb") as f:
        pickle.dump(macro_model, f)

In [18]:
len(macro_models[0].learning_firms)

100